# Prepare bird attribute data

This notebook does the complete workflow using only files in `data/separated_actual_data_required`: it builds the binary feature matrix `X` and species target vector `y`, creates a reproducible stratified 70/30 train/test split, and saves the four arrays as CSV files in `data/processed/`.

Expected dimensions:

| Array | Shape | Meaning |
| --- | --- | --- |
| `X` | `(11788, 312)` | Images by binary bird attributes |
| `y` | `(11788,)` | One-based bird species IDs |
| `X_train`, `X_test` | `(8251, 312)`, `(3537, 312)` | Training and test features |
| `y_train`, `y_test` | `(8251,)`, `(3537,)` | Training and test species IDs |

## Setup

Run this notebook with the project's Python environment. The project root is detected whether Jupyter starts in the repository root or in `notebooks/`. No separate Python script is needed.

In [1]:
from pathlib import Path
import numpy as np
from sklearn.model_selection import train_test_split

working_directory = Path.cwd().resolve()
project_candidates = (working_directory, working_directory.parent)
PROJECT_ROOT = next(
    (path for path in project_candidates if (path / "data" / "separated_actual_data_required").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate data/separated_actual_data_required")

DATA_DIR = PROJECT_ROOT / "data" / "separated_actual_data_required"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
NUM_IMAGES = 11788
NUM_ATTRIBUTES = 312
NUM_CLASSES = 200
print(f"Prepared data folder: {DATA_DIR}")

Prepared data folder: C:\Users\prajw\OneDrive\Desktop\Bird-Species-Identification\data\separated_actual_data_required


## 1. Check the required data files

This workflow uses the four files already present in the requested folder; it does not copy or read data from another folder. `image_features.txt` contains attribute records (`image_id`, `attribute_id`, and `is_present`); an image/attribute pair can have multiple judgments. The feature matrix is built from those records in the next step.

```text
data/separated_actual_data_required/
├── features/
│   ├── image_features.txt
│   └── feature_names.txt
└── labels/
    ├── image_labels.txt
    └── class_names.txt
```

In [2]:
print("Using files from:", DATA_DIR.relative_to(PROJECT_ROOT))
for relative_path in (
    "features/image_features.txt",
    "features/feature_names.txt",
    "labels/image_labels.txt",
    "labels/class_names.txt",
):
    path = DATA_DIR / relative_path
    if not path.is_file():
        raise FileNotFoundError(f"Required prepared-data file not found: {path}")
    print(f"- {path.relative_to(DATA_DIR)}")

Using files from: data\separated_actual_data_required
- features\image_features.txt
- features\feature_names.txt
- labels\image_labels.txt
- labels\class_names.txt


## 2. Create the feature matrix `X`

The feature file can contain repeated judgments for an image/attribute pair. This notebook uses a majority vote (a tie counts as present) to create one binary value per pair. This produces 11,788 image rows and 312 attribute columns.

In [3]:
feature_names_path = DATA_DIR / "features" / "feature_names.txt"
feature_names_by_id = {}
with feature_names_path.open(encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        fields = line.split(maxsplit=1)
        if len(fields) != 2:
            raise ValueError(f"Malformed feature name at line {line_number}")
        feature_id = int(fields[0])
        if not 1 <= feature_id <= NUM_ATTRIBUTES or feature_id in feature_names_by_id:
            raise ValueError(f"Invalid or duplicate feature ID {feature_id}")
        feature_names_by_id[feature_id] = fields[1].strip()
if len(feature_names_by_id) != NUM_ATTRIBUTES:
    raise ValueError(f"Expected {NUM_ATTRIBUTES} feature names")

present_counts = np.zeros((NUM_IMAGES, NUM_ATTRIBUTES), dtype=np.uint8)
annotation_counts = np.zeros_like(present_counts)
feature_file = DATA_DIR / "features" / "image_features.txt"
with feature_file.open(encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        fields = line.split()
        if len(fields) < 3:
            raise ValueError(f"Malformed feature annotation at line {line_number}")
        image_id, feature_id, present = map(int, fields[:3])
        if not 1 <= image_id <= NUM_IMAGES or not 1 <= feature_id <= NUM_ATTRIBUTES:
            raise ValueError(f"Invalid image/feature ID at line {line_number}")
        if present not in (0, 1):
            raise ValueError(f"Feature value must be 0 or 1 at line {line_number}")
        present_counts[image_id - 1, feature_id - 1] += present
        annotation_counts[image_id - 1, feature_id - 1] += 1
if np.any(annotation_counts == 0):
    raise ValueError("Some image/feature pairs have no annotations")
X = (2 * present_counts >= annotation_counts).astype(np.uint8)

assert X.shape == (11788, 312), f"Unexpected feature shape: {X.shape}"
assert set(np.unique(X)).issubset({0, 1}), "Features must be binary"

print("X shape:", X.shape)
print("Unique feature values:", np.unique(X))

X shape: (11788, 312)
Unique feature values: [0 1]


## 3. Create the target vector `y`

`image_labels.txt` provides one one-based species ID per image, so `y` has shape `(11788,)`. `class_names.txt` maps each ID to its species name; for example, the species name for `y[0]` is `class_names[y[0] - 1]`.

In [4]:
class_names_by_id = {}
with (DATA_DIR / "labels" / "class_names.txt").open(encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        fields = line.split(maxsplit=1)
        if len(fields) != 2:
            raise ValueError(f"Malformed class name at line {line_number}")
        class_id = int(fields[0])
        if not 1 <= class_id <= NUM_CLASSES or class_id in class_names_by_id:
            raise ValueError(f"Invalid or duplicate class ID {class_id}")
        class_names_by_id[class_id] = fields[1].strip()
if len(class_names_by_id) != NUM_CLASSES:
    raise ValueError(f"Expected {NUM_CLASSES} class names")
class_names = [class_names_by_id[class_id] for class_id in range(1, NUM_CLASSES + 1)]

y = np.zeros(NUM_IMAGES, dtype=np.int16)
seen_image_ids = np.zeros(NUM_IMAGES, dtype=bool)
with (DATA_DIR / "labels" / "image_labels.txt").open(encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        fields = line.split()
        if len(fields) != 2:
            raise ValueError(f"Malformed image label at line {line_number}")
        image_id, class_id = map(int, fields)
        if not 1 <= image_id <= NUM_IMAGES or seen_image_ids[image_id - 1]:
            raise ValueError(f"Invalid or duplicate image ID {image_id}")
        if not 1 <= class_id <= NUM_CLASSES:
            raise ValueError(f"Invalid class ID {class_id} at line {line_number}")
        y[image_id - 1] = class_id
        seen_image_ids[image_id - 1] = True
if not np.all(seen_image_ids):
    raise ValueError("Expected exactly one label for every image")
assert y.shape == (11788,), f"Unexpected label shape: {y.shape}"
assert len(class_names) == 200

print("y shape:", y.shape)
print("Number of species:", len(np.unique(y)))
print("First image class:", y[0], "->", class_names[y[0] - 1])
print("First five attributes:", [line.split(maxsplit=1)[1].strip() for line in (DATA_DIR / "features" / "feature_names.txt").read_text(encoding="utf-8").splitlines()[:5]])

y shape: (11788,)
Number of species: 200
First image class: 1 -> 001.Black_footed_Albatross
First five attributes: ['has_bill_shape::curved_(up_or_down)', 'has_bill_shape::dagger', 'has_bill_shape::hooked', 'has_bill_shape::needle', 'has_bill_shape::hooked_seabird']


## 4. Split the data into training and testing sets

A reproducible, class-stratified split reserves 30% of the images for testing and 70% for training. With 11,788 samples this yields 8,251 training samples and 3,537 testing samples.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

assert X_train.shape == (8251, 312), X_train.shape
assert X_test.shape == (3537, 312), X_test.shape
assert y_train.shape == (8251,), y_train.shape
assert y_test.shape == (3537,), y_test.shape

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)

X_train: (8251, 312)
X_test:  (3537, 312)
y_train: (8251,)
y_test:  (3537,)


The arrays `X_train`, `X_test`, `y_train`, and `y_test` are now ready for a classifier. The class IDs remain one-based to match the prepared label files.

## 5. Save the train/test arrays as CSV files

The feature matrices are saved as comma-separated rows with 312 columns. Each label vector is saved as a single-column CSV. Files contain no header or index, and the output folder is created if it does not already exist.

In [6]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
split_arrays = {"X_train": X_train, "X_test": X_test, "y_train": y_train, "y_test": y_test}
for name, array in split_arrays.items():
    csv_array = array.reshape(-1, 1) if array.ndim == 1 else array
    np.savetxt(PROCESSED_DIR / f"{name}.csv", csv_array, delimiter=",", fmt="%d")

for name, array in split_arrays.items():
    csv_path = PROCESSED_DIR / f"{name}.csv"
    loaded_array = np.loadtxt(csv_path, delimiter=",", dtype=array.dtype)
    if array.ndim == 1:
        loaded_array = loaded_array.reshape(-1)
    assert np.array_equal(loaded_array, array), f"CSV verification failed: {csv_path}"
    print(f"{csv_path.relative_to(PROJECT_ROOT)}: shape={array.shape}")

data\processed\X_train.csv: shape=(8251, 312)
data\processed\X_test.csv: shape=(3537, 312)
data\processed\y_train.csv: shape=(8251,)
data\processed\y_test.csv: shape=(3537,)
